# 準備

In [ ]:
# ライブラリのインポート

import pandas as pd
import seaborn as sns
import matplotlib as plt
import numpy as np
from pytest import skip

from sklearn.model_selection import train_test_split

import lightgbm as lgb

In [ ]:
# データの読み込み

data_train = pd.read_csv('train.csv')
data_test = pd.read_csv('test.csv')
data_submission = pd.read_csv('sample_submission.csv')

test_id = data_test['row_id'] # 結果の提出時に使用

data_test.tail()


,row_id,date,country,store,product
6565,32863,2019-12-31,Sweden,KaggleMart,Kaggle Hat
6566,32864,2019-12-31,Sweden,KaggleMart,Kaggle Sticker
6567,32865,2019-12-31,Sweden,KaggleRama,Kaggle Mug
6568,32866,2019-12-31,Sweden,KaggleRama,Kaggle Hat
6569,32867,2019-12-31,Sweden,KaggleRama,Kaggle Sticker


In [459]:
data_train.tail()

# 教師データ：3商品を2店で3カ国で売った4年間（2015-2818）の売り上げ個数データ
# テストデータ：同じくの2019年データを予測するabs

,row_id,date,country,store,product,num_sold
26293,26293,2018-12-31,Sweden,KaggleMart,Kaggle Hat,823
26294,26294,2018-12-31,Sweden,KaggleMart,Kaggle Sticker,250
26295,26295,2018-12-31,Sweden,KaggleRama,Kaggle Mug,1004
26296,26296,2018-12-31,Sweden,KaggleRama,Kaggle Hat,1441
26297,26297,2018-12-31,Sweden,KaggleRama,Kaggle Sticker,388


In [460]:
data_submission.tail()

,row_id,num_sold
6565,32863,100
6566,32864,100
6567,32865,100
6568,32866,100
6569,32867,100


In [461]:
data_train.info()
data_test.info()

# Nullなし、欠損値処理はしない

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26298 entries, 0 to 26297
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   row_id    26298 non-null  int64 
 1   date      26298 non-null  object
 2   country   26298 non-null  object
 3   store     26298 non-null  object
 4   product   26298 non-null  object
 5   num_sold  26298 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 1.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6570 entries, 0 to 6569
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   row_id   6570 non-null   int64 
 1   date     6570 non-null   object
 2   country  6570 non-null   object
 3   store    6570 non-null   object
 4   product  6570 non-null   object
dtypes: int64(1), object(4)
memory usage: 256.8+ KB


In [462]:
# dataを眺める
import ydata_profiling

data_train.profile_report()

# Storeとnum_sold、Productとnum_soldは若干正の相関あり

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 150.74it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [463]:
# 教師データとテストデータの結合
data_all = pd.concat([data_train, data_test], sort = False)
data_all.tail()

,row_id,date,country,store,product,num_sold
6565,32863,2019-12-31,Sweden,KaggleMart,Kaggle Hat,NaN
6566,32864,2019-12-31,Sweden,KaggleMart,Kaggle Sticker,NaN
6567,32865,2019-12-31,Sweden,KaggleRama,Kaggle Mug,NaN
6568,32866,2019-12-31,Sweden,KaggleRama,Kaggle Hat,NaN
6569,32867,2019-12-31,Sweden,KaggleRama,Kaggle Sticker,NaN


In [464]:
# カテゴリデータの変換
from numpy import dtype


data_all["country"].replace(["Finland", "Norway", "Sweden"], [0, 1, 2], inplace=True)
data_all["store"].replace(["KaggleMart", "KaggleRama"], [0, 1], inplace=True)
data_all["product"].replace(["Kaggle Mug", "Kaggle Hat", "Kaggle Sticker"], [0, 1, 2], inplace=True)

# なんでか数字にしたのにObjectになってるからintにした
data_all["country"] = data_all["country"].astype("int")
data_all["store"] = data_all["store"].astype("int")
data_all["product"] = data_all["product"].astype("int")

data_all.head()

/var/folders/lj/rcsyr8rj4gl5yjh1p78gq8rr0000gn/T/ipykernel_4948/1967265776.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_all["country"].replace(["Finland", "Norway", "Sweden"], [0, 1, 2], inplace=True)
/var/folders/lj/rcsyr8rj4gl5yjh1p78gq8rr0000gn/T/ipykernel_4948/1967265776.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object 

,row_id,date,country,store,product,num_sold
0,0,2015-01-01,0,0,0,329.0
1,1,2015-01-01,0,0,1,520.0
2,2,2015-01-01,0,0,2,146.0
3,3,2015-01-01,0,1,0,572.0
4,4,2015-01-01,0,1,1,911.0


In [465]:
data_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32868 entries, 0 to 6569
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   row_id    32868 non-null  int64  
 1   date      32868 non-null  object 
 2   country   32868 non-null  int64  
 3   store     32868 non-null  int64  
 4   product   32868 non-null  int64  
 5   num_sold  26298 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 1.8+ MB


In [466]:
# # カテゴリーデータをダミー変数で変換（ダミー変数で変換する場合）
# cat_labels = ['country', 'store', 'product']  # カテゴリ変数のラベルを指定
# # dtype = int typeがboolのTrue/False表示になったので
# data_all = pd.get_dummies(data_all, columns = cat_labels, dtype = int) 
# data_all.head()

In [467]:
# 時系列データの変換（Objectになっていたのでintに）
# かつ新しい特徴量の作成
data_all["date"] = pd.to_datetime(data_all["date"])
data_all["day"] = data_all.date.dt.day
data_all["month"] = data_all.date.dt.month
data_all["year"] = data_all.date.dt.year
data_all["weekday"] = data_all.date.dt.weekday # 何曜日

data_all.head()


,row_id,date,country,store,product,num_sold,day,month,year,weekday
0,0,2015-01-01,0,0,0,329.0,1,1,2015,3
1,1,2015-01-01,0,0,1,520.0,1,1,2015,3
2,2,2015-01-01,0,0,2,146.0,1,1,2015,3
3,3,2015-01-01,0,1,0,572.0,1,1,2015,3
4,4,2015-01-01,0,1,1,911.0,1,1,2015,3


In [468]:
# 分析にいらない特徴量はdrop
data_all.drop(['date', 'row_id'], axis=1, inplace=True)
data_all.head()
data_all.info()
# int32でも±21億まで数えれるから問題なし。

<class 'pandas.core.frame.DataFrame'>
Index: 32868 entries, 0 to 6569
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   country   32868 non-null  int64  
 1   store     32868 non-null  int64  
 2   product   32868 non-null  int64  
 3   num_sold  26298 non-null  float64
 4   day       32868 non-null  int32  
 5   month     32868 non-null  int32  
 6   year      32868 non-null  int32  
 7   weekday   32868 non-null  int32  
dtypes: float64(1), int32(4), int64(3)
memory usage: 1.8 MB


In [469]:
data_all.tail()

,country,store,product,num_sold,day,month,year,weekday
6565,2,0,1,NaN,31,12,2019,1
6566,2,0,2,NaN,31,12,2019,1
6567,2,1,0,NaN,31,12,2019,1
6568,2,1,1,NaN,31,12,2019,1
6569,2,1,2,NaN,31,12,2019,1


# データの分割

In [470]:
# 結合したデータを、再度、教師データと、テストデータに分ける
data_train = data_all[:len(data_train)]
data_test = data_all[len(data_train):]

In [471]:
data_train.tail()

,country,store,product,num_sold,day,month,year,weekday
26293,2,0,1,823.0,31,12,2018,0
26294,2,0,2,250.0,31,12,2018,0
26295,2,1,0,1004.0,31,12,2018,0
26296,2,1,1,1441.0,31,12,2018,0
26297,2,1,2,388.0,31,12,2018,0


In [472]:
data_test.tail()

,country,store,product,num_sold,day,month,year,weekday
6565,2,0,1,NaN,31,12,2019,1
6566,2,0,2,NaN,31,12,2019,1
6567,2,1,0,NaN,31,12,2019,1
6568,2,1,1,NaN,31,12,2019,1
6569,2,1,2,NaN,31,12,2019,1


In [473]:
# モデル作成準備、目的変数yと説明変xを学習させ、テストデータを食わせる。
y_data_train = data_train['num_sold'] # 教師
x_data_train = data_train.drop('num_sold', axis=1) # 訓練データ
x_data_test = data_test.drop('num_sold', axis=1) # テストデータも同じく説明変数だけにする

y_data_train.tail()

26293     823.0
26294     250.0
26295    1004.0
26296    1441.0
26297     388.0
Name: num_sold, dtype: float64

In [474]:
# 訓練用データを訓練用と検証用に分割します。
# 目的変数yと説明変xを学習させる。
from sklearn.model_selection import train_test_split

x_data_train, x_data_valid, y_data_train, y_data_valid = train_test_split(x_data_train, y_data_train, test_size=0.3)
# random_state=0, stratify=y_data_trainは、分類。今回は販売数の予測なので回帰。

# モデルの訓練

In [ ]:
from lightgbm import early_stopping, log_evaluation


categorical_features = ["country", "store", "product", 'month', 'weekday'] 
# Day以外は少ないのでカテゴリとしている
# Yearはカテゴリ変数にすると季節性が学習できない

# LightGBMのライブラリでは、「LGBMRegressor」により回帰問題を扱うことができます。
# ちょっとこれはTitanicのLightGBMとはやり方の違いがわからんし、精度工場についても学ばなあかんあ

# モデルの構築
model = lgb.LGBMRegressor() # 今回はハイパラ設定はしない
# 目的変数yと説明変xを学習させる。
model = model.fit(x_data_train, y_data_train,
                  eval_set=[(x_data_train, y_data_train), (x_data_valid, y_data_valid)],
                  categorical_feature = categorical_features,
                  callbacks=[
                      early_stopping(stopping_rounds=100),
                      log_evaluation(period=100)
                  ]
)



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000179 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 69
[LightGBM] [Info] Number of data points in the train set: 18408, number of used features: 7
[LightGBM] [Info] Start training from score 388.945893
Training until validation scores don't improve for 100 rounds
[100]	training's l2: 1037.59	valid_1's l2: 1312.9
Did not meet early stopping. Best iteration is:
[100]	training's l2: 1037.59	valid_1's l2: 1312.9


In [476]:
# 誤差の推移を表示
%matplotlib inline

lgb.plot_metric(model)

# 表示されんな

<Axes: title={'center': 'Metric during training'}, xlabel='Iterations', ylabel='l2'>

# 提出用データ作成

In [477]:
# 訓練用のモデルを使って、テストデータの予測を行う
y_data_pred = model.predict(x_data_test)
y_data_pred[:5]

array([301.313108  , 447.56706158, 148.98055544, 556.19001526,
       848.24501535])

In [478]:
# 形式を整える
num_sold_test = pd.Series(y_data_pred, name="num_sold")
submit = pd.concat([test_id, num_sold_test], axis=1)

# 提出用のcsvファイルを保存
submit.to_csv("jan2022_lgbregression_submission.csv", index=False)

submit

,row_id,num_sold
0,26298,301.313108
1,26299,447.567062
2,26300,148.980555
3,26301,556.190015
4,26302,848.245015
...,...,...
6565,32863,787.297314
6566,32864,229.725106
6567,32865,971.487272
6568,32866,1425.109931
